#  Hiding future words with causal attention

- `causal Attention`: it is the process of restricting the model to update the attention score based on the future words or texts.

- The causal aspect involves modifying the attention mechanism to prevent the model from accessing future information in the sequence, which is crucial for tasks like language modeling, where each word prediction should only depend on previous words.

- the self-attention mechanism is to only for the tokens that appear prior to the current position when predicting the next token in a
sequence.

- we know that casual attention is also masked attention, which is to predict the next token only based on the previous and current tokens,  but not future tokens. but in the self attention which allows access to the entire inputsequence at once. that is the disadvantage.

- now to obtain the casual attention we need to mask or hide the future tokens in the input.

- masking the weights diagoanlly in the attention_weights, can achieve the this.

  ![13. Weights Masking](../images/13.weightsMasking.png)


  ![14. Weights Masking Flow](../images/14.maskFlow.png)

- we need attention_weights to mask. we use our previous v2 of simple attention.
  

In [4]:
import torch
inputs = torch.tensor(
 [[0.43, 0.15, 0.89], # Your (x^1)
 [0.55, 0.87, 0.66], # journey (x^2)
 [0.57, 0.85, 0.64], # starts (x^3)
 [0.22, 0.58, 0.33], # with (x^4)
 [0.77, 0.25, 0.10], # one (x^5)
 [0.05, 0.80, 0.55]] # step (x^6)
)

In [5]:
import torch.nn as nn
class Self_Attention_v2(nn.Module):
    def __init__(self, d_in, d_out,qkv_bias=False):
        super().__init__()
        # initialise the weights but with nn.linear
        self.wq = nn.Linear(d_in,d_out, bias=qkv_bias)
        self.wk = nn.Linear(d_in,d_out, bias=qkv_bias)
        self.wv = nn.Linear(d_in,d_out, bias=qkv_bias)

    def forward(self, inputs):
        # get all keys, values and queries from input based on the weights
        queries = self.wq(inputs)
        keys = self.wk(inputs)
        values = self.wv(inputs)

        # attetion weights by keys and queries
        attention_scores = queries @ keys.T

        # attention scores after normalisation
        attention_weights = torch.softmax(attention_scores/keys.shape[-1]**0.5, dim=-1)

        # final context vectors
        context_vectors = attention_weights @ values

        return context_vectors

In [10]:
torch.manual_seed(789)
d_in = inputs.shape[1]
d_out =2
sa_v2 = Self_Attention_v2(d_in, d_out)
queries = sa_v2.wq(inputs)
keys = sa_v2.wk(inputs)

attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)
attn_scores.shape

tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


torch.Size([6, 6])

- creating a mask above upper diagonal zero for a matrix.
- We can implement the second step using PyTorch’s `tril` function to create a mask where the values above the diagonal are zero

In [9]:
context_length = attn_scores.shape[0] # mat dimensions for creating a new mask matrix
simple_mask = torch.tril(torch.ones(context_length, context_length))
simple_mask

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])

In [13]:
masked = simple_mask * attn_weights
masked


tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)

- renormalize the attention weights to sum up to 1 again in each row. 

In [14]:
row_sums = masked.sum(dim=-1, keepdim=True)
masked_norm = masked/row_sums
row_sums,masked_norm


(tensor([[0.1921],
         [0.3700],
         [0.5357],
         [0.6775],
         [0.8415],
         [1.0000]], grad_fn=<SumBackward1>),
 tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
         [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
         [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
         [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
        grad_fn=<DivBackward0>))

# information leakage

- There are some chances of the future tokens still influencing the current available in masked, bcz previously attention weights are calculated using the softmax with all the tokens that includes the future tokens too.

- so if we renormalise the softmaxes over the unmasked token subsets then it fulfills the future tokens information leakage.

- for doing there is most efficient way is to , while masking with zero, do it with the -infinity, because the softmax is in the power of the e, so -infinity will ends up with 0 itself.

  ![15. Efficient Masking Flow](../images/15.masking_best.png)

  
- `The softmax function converts its inputs into a probability distribution.` - this is y i am reading probability from years :😂

In [15]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
prev = masked
print(f"weights before masking {prev}")
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
masked

weights before masking tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)


tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackward0>)

In [16]:
#now apply the softmax
attn_weights = torch.softmax(masked/keys.shape[0], dim=1)
attn_weights
# i can see here weights are modified after removing the future tokens supports in the weights after normalisation

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5122, 0.4878, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3441, 0.3279, 0.3280, 0.0000, 0.0000, 0.0000],
        [0.2560, 0.2491, 0.2492, 0.2457, 0.0000, 0.0000],
        [0.2040, 0.1996, 0.1997, 0.1973, 0.1993, 0.0000],
        [0.1727, 0.1667, 0.1667, 0.1637, 0.1667, 0.1634]],
       grad_fn=<SoftmaxBackward0>)

#  Masking additional attention weights with dropout

- why actually we need dropout?
  
- due to the problem of the`overfitting, model performs well on the trained data but struggles on new and unseen data`, to fix this issue we need the dropout.

- It is a regularisation method that `prevents overfitting, it randomly drops some nodes during nueral network training`.

# what is overfitting?
- The model which acutally performs well on training with known data, but when new questions appears they struggle generate the new data.

- It is due to multiple layers and nodes in the nueral networks they learn some patterns previously.
  
- `Overfitting in neural networks happens because of the network’s complexity` — multiple layers and nodes, `all capturing intricate patterns`. When overfitting, `a model can memorize noise or irrelevant patterns, which doesn’t help when it encounters new data`.

# what is dropout?
- it is the process of disabling randomly some nuerons while training, it avoids too much dependency on particular nuerons .even new nodes learn the pattern to generate. that makes to perform better on new data.

- `To better understand, let’s compare a neural network using dropout to a group study session where different students (neurons) are randomly asked to leave the room during each session. The remaining students must still work together to solve problems, which strengthens each person’s understanding of the material.`

- during training: dropout rate is 50%(disables nuerons), at each iteration.`This effectively creates a new, smaller neural network with fewer neurons for that iteration. As a result, the model learns to work without depending on any one specific neuron`, which reduces overfitting.

- during testing: all nuerons are active.

# example of dropout

- Let’s look at a simple neural network with three layers: an input layer, a hidden layer with four neurons, and an output layer. Suppose we set the dropout rate at 0.5.

- Epoch 1: Two neurons in the hidden layer are randomly disabled.
- Epoch 2: A different set of neurons might be disabled, such as one neuron in the input layer and one in the hidden layer.
- Epoch 3: Again, a different configuration of active neurons is used.

- `Each epoch has a unique configuration of active and inactive neurons, which means the network is training in a slightly different way each time. Over multiple epochs, this randomness leads to a robust model that doesn’t rely on any single neuron`.

- dropout in the attention mechanism is typically applied at two specific times: after calculating the attention weights or after applying the attention weights to the value vectors.

In [15]:
import torch
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5) # dropped or disabled 50%
ones = torch.ones(6,6)
ones, dropout(ones)

(tensor([[1., 1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1., 1.]]),
 tensor([[2., 2., 0., 2., 2., 0.],
         [0., 0., 0., 2., 0., 2.],
         [2., 2., 2., 2., 0., 2.],
         [0., 2., 2., 0., 0., 2.],
         [0., 2., 0., 2., 0., 2.],
         [0., 2., 2., 2., 2., 0.]]))

- during dropout, to compensate the disabled, active nuerons are scaled up. here doubled.
- This scaling is crucial to maintain the overall balance of the attention weights, ensuring that the average influence of the attention mechanism remains consistent during both the training and inference phases.
-  ![16. Drop out](../images/16.dropout.png)

In [16]:
# let's apply dropout to our attention matrix 
torch.manual_seed(123)
dropout(attn_weights)

tensor([[0.3843, 0.3293, 0.0000, 0.3100, 0.3442, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.2992, 0.0000, 0.2955],
        [0.4071, 0.3318, 0.3325, 0.2996, 0.0000, 0.2961],
        [0.0000, 0.3334, 0.3337, 0.0000, 0.0000, 0.3128],
        [0.0000, 0.3337, 0.0000, 0.3177, 0.0000, 0.3169],
        [0.0000, 0.3327, 0.3331, 0.3084, 0.3331, 0.0000]],
       grad_fn=<MulBackward0>)

 # Implementing a compact causal attention class

 - making the self attention class to masking the masking and dropout, btw it also processes the batches as in dataloader.

 - to simulate it we are randomly using the dummy batches here.

 - `torch.stack` Concatenates a sequence of tensors along a new dimension. (x, x) => x repeats 2 times

In [17]:
batches = torch.stack((inputs, inputs), dim=0)
batches, batches.shape

(tensor([[[0.4300, 0.1500, 0.8900],
          [0.5500, 0.8700, 0.6600],
          [0.5700, 0.8500, 0.6400],
          [0.2200, 0.5800, 0.3300],
          [0.7700, 0.2500, 0.1000],
          [0.0500, 0.8000, 0.5500]],
 
         [[0.4300, 0.1500, 0.8900],
          [0.5500, 0.8700, 0.6600],
          [0.5700, 0.8500, 0.6400],
          [0.2200, 0.5800, 0.3300],
          [0.7700, 0.2500, 0.1000],
          [0.0500, 0.8000, 0.5500]]]),
 torch.Size([2, 6, 3]))

# A compact causal attention class

In [18]:
class CasualAttention(nn.Module):
    def __init__(self, d_in, d_out,dropout,context_length, qkv_bias = False):
        super().__init__()
        # initialise the weights but with nn.linear
        self.d_out = d_out
        self.wq = nn.Linear(d_in,d_out, bias=qkv_bias)
        self.wk = nn.Linear(d_in,d_out, bias=qkv_bias)
        self.wv = nn.Linear(d_in,d_out, bias=qkv_bias)

        #dropouts and context length
        self.dropout = nn.Dropout(dropout)

        # masks too.
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length),diagonal=1))

    def forward(self, inputs):
        batches, num_tokens, d_in = inputs.shape

        #keys, queries, values
        keys = self.wk(inputs)
        queries = self.wq(inputs)
        values = self.wv(inputs)

        # establishing relationships 
        attention_scores = queries @keys.transpose(1,2)

        # normalise scores and mask weights
        attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf) # context length

        #attention weights
        attn_weights = torch.softmax(attn_scores/keys.shape[-1]**0.5, dim=-1)

        #dropout
        attn_weights = self.dropout(attn_weights)

        # context vectors
        context_vectors =  attn_weights @values

        return context_vectors
    

- The use of `register_buffer` in PyTorch is not strictly necessary for all use cases but offers several advantages here.

- For instance, when we use the CausalAttention class in our LLM, buffers are automatically moved to the appropriate device (CPU or GPU) along with our model, which will be relevant when training our LLM.

- This means we don’t need to manually ensure these tensors are on the same device as your model parameters, avoiding device mismatch errors.

In [19]:
ca = CasualAttention(d_in, d_out, 0.5, batches.shape[1])
context_vec = ca(batches)
context_vec

tensor([[[ 0.0000,  0.0000],
         [-1.0668,  1.0822],
         [-0.7335,  0.7469],
         [-0.7296,  0.8279],
         [-0.6810,  0.7226],
         [-0.5926,  0.6579]],

        [[ 0.0000,  0.0000],
         [-1.0668,  1.0822],
         [-0.7335,  0.7469],
         [-0.7296,  0.8279],
         [-0.6810,  0.7226],
         [-0.5926,  0.6579]]], grad_fn=<CloneBackward0>)